# Quote Policy Optimizer: Standalone and Supported Market Making

This notebook turns the previous research into a decision rule for whether we should quote, how long quotes may rest, and what evidence is missing before paper or live trading.

The key distinction is:

- **Standalone market making**: the passive quote has positive expected value from execution economics alone: fill probability, markout after fill, fee or rebate, queue position, latency, and cancel quality.
- **Supported market making**: the passive quote is acceptable only because another measured portfolio edge supports it: fair value, inventory reduction, hedge or basis spread, venue rebate economics, or queue/latency advantage.

Supported does not mean we hide a bad quote. It means the quote must still clear an explicit portfolio EV equation after execution risk and model-risk haircuts.


## Portfolio EV Definition

For side `s`, distance `d`, and order size `q`, standalone quote value is:

$$
EV^{standalone}_{t,s,d}
= P(fill_{t,s,d})
\left(E[markout_{t+h,s,d}\mid fill] + r_{maker} - c_{fees}\right)
- C_{queue} - C_{latency} - C_{ops}
$$

A supported market-making quote adds a measured portfolio support term:

$$
EV^{portfolio}_{t,s,d}
= EV^{standalone}_{t,s,d}
+ P(fill_{t,s,d})\left(\eta_s(F_t-M_t) - \Delta Risk(q_t, s, q) + H_t\right)
- C_{model}
$$

where:

- `eta_s = +1` for a bid and `-1` for an ask.
- `F_t-M_t` is fair-value edge relative to current mid.
- `Delta Risk` is the inventory risk change from the fill.
- `H_t` is hedge, basis, funding, or cross-venue support.
- `C_model` is a haircut for weak sample size, unstable features, and simulation mismatch.

A quote is **standalone** only if `EV_standalone > 0` under event replay. A quote is **supported** if standalone EV is not enough, but portfolio EV is positive and the support term is observable, auditable, and available before the quote is sent.


## Selection Rules

To avoid p-hacking, this notebook does not pick the highest realized PnL row and call it a strategy. It separates gates:

1. **Paper gate**: useful enough to paper trade and collect ack/fill/cancel logs.
2. **Live gate**: strict enough to risk capital.
3. **Research only**: positive-looking rows that fail capacity, L2 coverage, stale resting time, or sample-size checks.

The main rule is that a supported market-making strategy must name its support before execution. If the support is fair value, we need fair-value edge at decision time. If the support is inventory, we need inventory state and risk reduction. If the support is queue, we need actual queue and cancel telemetry.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display

def find_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "notebooks" / "strategy_execution_replay.ipynb").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing notebooks/strategy_execution_replay.ipynb")


PROJECT_ROOT = find_project_root()
strategy_notebook_env = os.environ.get("MODL_STRATEGY_REPLAY_NOTEBOOK")
if strategy_notebook_env:
    STRATEGY_NOTEBOOK = Path(strategy_notebook_env).expanduser()
    if not STRATEGY_NOTEBOOK.is_absolute():
        STRATEGY_NOTEBOOK = (PROJECT_ROOT / STRATEGY_NOTEBOOK).resolve()
else:
    STRATEGY_NOTEBOOK = PROJECT_ROOT / "notebooks" / "strategy_execution_replay.ipynb"
FEATURE_ROOT = Path(os.environ.get("MODL_WS_FEATURE_DIR", "/mnt/burner-archive/ws_features")).expanduser()
BAR_MINUTES = int(os.environ.get("MODL_BAR_MINUTES", "5"))

BASE_LATENCY_MS = int(os.environ.get("MODL_QUOTE_OPT_BASE_LATENCY_MS", "200"))
BASE_REBATE_BPS = float(os.environ.get("MODL_QUOTE_OPT_BASE_REBATE_BPS", "2.5"))

PAPER_MIN_FILL_RATE = float(os.environ.get("MODL_QUOTE_OPT_PAPER_MIN_FILL_RATE", "0.02"))
PAPER_MIN_FULL_FILLS = int(os.environ.get("MODL_QUOTE_OPT_PAPER_MIN_FULL_FILLS", "3"))
PAPER_MAX_TTL_MS = int(os.environ.get("MODL_QUOTE_OPT_PAPER_MAX_TTL_MS", "1800000"))

LIVE_MIN_FILL_RATE = float(os.environ.get("MODL_QUOTE_OPT_LIVE_MIN_FILL_RATE", "0.02"))
LIVE_MIN_FULL_FILLS = int(os.environ.get("MODL_QUOTE_OPT_LIVE_MIN_FULL_FILLS", "10"))
LIVE_MIN_L2_COVERAGE = float(os.environ.get("MODL_QUOTE_OPT_LIVE_MIN_L2_COVERAGE", "0.20"))
LIVE_MAX_TTL_MS = int(os.environ.get("MODL_QUOTE_OPT_LIVE_MAX_TTL_MS", "300000"))
LIVE_MAX_MEDIAN_FILL_DELAY_MS = int(os.environ.get("MODL_QUOTE_OPT_LIVE_MAX_FILL_DELAY_MS", "300000"))
LIVE_MAX_MEAN_QUEUE_AHEAD_QTY = float(os.environ.get("MODL_QUOTE_OPT_LIVE_MAX_QUEUE_AHEAD_QTY", "0.75"))

STALE_PENALTY_BPS_PER_MIN = float(os.environ.get("MODL_QUOTE_OPT_STALE_PENALTY_BPS_PER_MIN", "0.05"))
QUEUE_PENALTY_BPS_PER_BTC = float(os.environ.get("MODL_QUOTE_OPT_QUEUE_PENALTY_BPS_PER_BTC", "0.25"))
COVERAGE_SHORTFALL_PENALTY_BPS = float(os.environ.get("MODL_QUOTE_OPT_COVERAGE_SHORTFALL_PENALTY_BPS", "5.0"))
FILL_SHORTFALL_PENALTY_BPS = float(os.environ.get("MODL_QUOTE_OPT_FILL_SHORTFALL_PENALTY_BPS", "25.0"))

SAVE_OUTPUTS = False

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 260)
pd.set_option("display.max_colwidth", 240)

PROJECT_ROOT, STRATEGY_NOTEBOOK, BASE_LATENCY_MS, BASE_REBATE_BPS


## Load Strategy Replay Outputs

The optimizer executes `strategy_execution_replay.ipynb` in a clean namespace. That gives us the same inventory decisions, fair-value support, fill model, event replay, L2 queue estimate, and scenario grid used in the prior notebook.


In [ ]:
def quiet_display(*args, **kwargs):
    return None


def execute_strategy_notebook(path: Path) -> dict[str, object]:
    original_display = ipd.display
    ipd.display = quiet_display
    namespace: dict[str, object] = {"__name__": "__strategy_replay__"}
    notebook = json.loads(path.read_text())
    original_cwd = Path.cwd()
    try:
        os.chdir(PROJECT_ROOT)
        for cell in notebook["cells"]:
            if cell.get("cell_type") != "code":
                continue
            source = "".join(cell.get("source", []))
            if not source.strip():
                continue
            exec(compile(source, f"{path}:{cell.get('id', 'cell')}", "exec"), namespace)
            plt.close("all")
    finally:
        os.chdir(original_cwd)
        ipd.display = original_display
    return namespace


strategy_ns = execute_strategy_notebook(STRATEGY_NOTEBOOK)

policy_input_summary = strategy_ns["policy_input_summary"].copy()
base_strategy_replay = strategy_ns["base_strategy_replay"].copy()
strategy_scenario_summary = strategy_ns["strategy_scenario_summary"].copy()
strategy_replay_by_ttl = strategy_ns["strategy_replay_by_ttl"].copy()
model_event_gap = strategy_ns["model_event_gap"].copy()

display(policy_input_summary)
display(strategy_scenario_summary.head(20))


## Strategy Angles

There are several legitimate ways to build a market-making strategy. The question is whether each one is standalone, supported, or not yet supported by the data.


In [ ]:
angle_scorecard = pd.DataFrame(
    [
        {
            "angle": "standalone passive spread/rebate capture",
            "support_source": "fill probability, post-fill markout, maker rebate, queue priority",
            "current_evidence": "weak for short TTL; event replay shows limited capacity once L2 queue is included",
            "required_next_evidence": "positive event EV with enough L2 queue-aware fills and stable paper/live slippage",
            "current_status": "research only",
        },
        {
            "angle": "fair-value supported quoting",
            "support_source": "forecasted fair value versus live mid",
            "current_evidence": "available in inventory quote model, but execution gate is still the binding constraint",
            "required_next_evidence": "quote closer or size smaller while preserving post-fill markout after L2 queue and latency",
            "current_status": "paper candidate after policy gate",
        },
        {
            "angle": "inventory-supported quoting",
            "support_source": "risk reduction when quote moves inventory toward target",
            "current_evidence": "bid-side inventory score is strongest in replay, but relies on long resting windows",
            "required_next_evidence": "paper logs showing fills reduce inventory risk without stale adverse selection",
            "current_status": "paper candidate, not live",
        },
        {
            "angle": "hedged or cross-venue supported market making",
            "support_source": "basis, funding, hedge spread, or portfolio covariance reduction",
            "current_evidence": "hedging research exists, but quote decisions are not yet joined to executable hedge state",
            "required_next_evidence": "joint quote plus hedge simulation with hedge slippage and funding/carry attribution",
            "current_status": "next research branch",
        },
        {
            "angle": "queue/latency supported quoting",
            "support_source": "fast ack/cancel, queue priority, low missed-fill rate",
            "current_evidence": "we have L2 snapshots, not actual ack/cancel/queue telemetry",
            "required_next_evidence": "paper trading logs with decision time, send time, ack time, cancel ack, fill events, and local market-data sequence",
            "current_status": "engineering prerequisite",
        },
    ]
)

display(angle_scorecard)


## Policy Frontier

The frontier converts replay metrics into policy decisions. We keep both raw event EV and an implementation-adjusted score. The adjusted score penalizes stale fills, queue size, and poor L2 coverage because those are exactly where historical passive simulations tend to be too optimistic.


In [ ]:
def add_policy_scores(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["ttl_minutes"] = out["ttl_ms"] / 60_000.0
    out["median_fill_delay_minutes"] = out["median_fill_delay_ms"] / 60_000.0
    out["median_fill_delay_minutes"] = out["median_fill_delay_minutes"].fillna(out["ttl_minutes"])
    out["stale_penalty_bps"] = out["median_fill_delay_minutes"].clip(lower=0.0) * STALE_PENALTY_BPS_PER_MIN
    out["queue_penalty_bps"] = out["mean_l2_queue_ahead_qty"].fillna(0.0).clip(lower=0.0) * QUEUE_PENALTY_BPS_PER_BTC
    out["coverage_shortfall"] = (LIVE_MIN_L2_COVERAGE - out["l2_price_coverage_rate"].fillna(0.0)).clip(lower=0.0)
    out["coverage_penalty_bps"] = out["coverage_shortfall"] * COVERAGE_SHORTFALL_PENALTY_BPS
    out["fill_shortfall"] = (LIVE_MIN_FILL_RATE - out["event_full_fill_rate_l2_queue"].fillna(0.0)).clip(lower=0.0)
    out["fill_penalty_bps"] = out["fill_shortfall"] * FILL_SHORTFALL_PENALTY_BPS
    out["implementation_score_bps"] = (
        out["event_ev_after_rebate_bps"].fillna(0.0)
        - out["stale_penalty_bps"]
        - out["queue_penalty_bps"]
        - out["coverage_penalty_bps"]
        - out["fill_penalty_bps"]
    )
    out["standalone_positive_model"] = out["mean_predicted_standalone_bps"].fillna(-np.inf) > 0.0
    out["portfolio_supported_model"] = out["mean_model_quote_score_bps"].fillna(-np.inf) > out["mean_predicted_standalone_bps"].fillna(-np.inf)
    out["support_gap_bps"] = out["mean_model_quote_score_bps"] - out["mean_predicted_standalone_bps"]
    out["paper_gate"] = (
        (out["orders"] >= 20)
        & (out["event_fills_l2_queue"] >= PAPER_MIN_FULL_FILLS)
        & (out["event_full_fill_rate_l2_queue"] >= PAPER_MIN_FILL_RATE)
        & (out["ttl_ms"] <= PAPER_MAX_TTL_MS)
        & (out["event_ev_after_rebate_bps"] > 0.0)
    )
    out["live_gate"] = (
        (out["orders"] >= 50)
        & (out["event_fills_l2_queue"] >= LIVE_MIN_FULL_FILLS)
        & (out["event_full_fill_rate_l2_queue"] >= LIVE_MIN_FILL_RATE)
        & (out["l2_price_coverage_rate"] >= LIVE_MIN_L2_COVERAGE)
        & (out["ttl_ms"] <= LIVE_MAX_TTL_MS)
        & (out["median_fill_delay_ms"].fillna(np.inf) <= LIVE_MAX_MEDIAN_FILL_DELAY_MS)
        & (out["mean_l2_queue_ahead_qty"].fillna(np.inf) <= LIVE_MAX_MEAN_QUEUE_AHEAD_QTY)
        & (out["implementation_score_bps"] > 0.0)
    )
    out["strategy_type"] = np.select(
        [
            out["standalone_positive_model"] & (out["event_ev_after_rebate_bps"] > 0.0),
            out["portfolio_supported_model"] & (out["event_ev_after_rebate_bps"] > 0.0),
            out["event_ev_after_rebate_bps"] > 0.0,
        ],
        ["standalone candidate", "supported candidate", "realized-only candidate"],
        default="not positive",
    )
    return out.sort_values("implementation_score_bps", ascending=False).reset_index(drop=True)


policy_frontier = add_policy_scores(strategy_scenario_summary)
frontier_cols = [
    "policy",
    "strategy_type",
    "latency_ms",
    "ttl_ms",
    "maker_rebate_bps",
    "orders",
    "event_fills_l2_queue",
    "event_full_fill_rate_l2_queue",
    "l2_price_coverage_rate",
    "mean_l2_queue_ahead_qty",
    "median_fill_delay_ms",
    "mean_predicted_standalone_bps",
    "mean_model_quote_score_bps",
    "support_gap_bps",
    "event_ev_after_rebate_bps",
    "implementation_score_bps",
    "paper_gate",
    "live_gate",
]

display(policy_frontier[frontier_cols].head(30))


## Gate Results

A live-ready result should pass the live gate without hand tuning. A paper result is allowed to be looser because its job is to validate simulation mismatch, not risk meaningful capital.


In [ ]:
gate_summary = pd.DataFrame(
    [
        {
            "gate": "paper",
            "passing_rows": int(policy_frontier["paper_gate"].sum()),
            "best_score_bps": policy_frontier.loc[policy_frontier["paper_gate"], "implementation_score_bps"].max() if policy_frontier["paper_gate"].any() else np.nan,
            "best_ev_bps": policy_frontier.loc[policy_frontier["paper_gate"], "event_ev_after_rebate_bps"].max() if policy_frontier["paper_gate"].any() else np.nan,
        },
        {
            "gate": "live",
            "passing_rows": int(policy_frontier["live_gate"].sum()),
            "best_score_bps": policy_frontier.loc[policy_frontier["live_gate"], "implementation_score_bps"].max() if policy_frontier["live_gate"].any() else np.nan,
            "best_ev_bps": policy_frontier.loc[policy_frontier["live_gate"], "event_ev_after_rebate_bps"].max() if policy_frontier["live_gate"].any() else np.nan,
        },
    ]
)

paper_candidates = policy_frontier[policy_frontier["paper_gate"]].copy()
live_candidates = policy_frontier[policy_frontier["live_gate"]].copy()
research_candidates = policy_frontier[
    (policy_frontier["event_ev_after_rebate_bps"] > 0.0) & ~policy_frontier["paper_gate"]
].copy()

display(gate_summary)
display(paper_candidates[frontier_cols].head(20))
display(live_candidates[frontier_cols].head(20))
display(research_candidates[frontier_cols].head(20))


## Constraint Sensitivity

This grid shows how fragile the best policy is as we tighten minimum fill rate, maximum TTL, and L2 coverage. If the best row disappears as soon as coverage is required, the strategy is not live-ready even if realized EV is positive.


In [ ]:
def best_under_constraints(min_fill_rate: float, max_ttl_ms: int, min_l2_coverage: float) -> dict[str, object]:
    eligible = policy_frontier[
        (policy_frontier["event_full_fill_rate_l2_queue"] >= min_fill_rate)
        & (policy_frontier["ttl_ms"] <= max_ttl_ms)
        & (policy_frontier["l2_price_coverage_rate"] >= min_l2_coverage)
        & (policy_frontier["event_ev_after_rebate_bps"] > 0.0)
    ].copy()
    if eligible.empty:
        return {
            "min_fill_rate": min_fill_rate,
            "max_ttl_ms": max_ttl_ms,
            "min_l2_coverage": min_l2_coverage,
            "eligible_rows": 0,
            "best_policy": None,
            "best_strategy_type": None,
            "best_ev_bps": np.nan,
            "best_score_bps": np.nan,
        }
    best = eligible.sort_values("implementation_score_bps", ascending=False).iloc[0]
    return {
        "min_fill_rate": min_fill_rate,
        "max_ttl_ms": max_ttl_ms,
        "min_l2_coverage": min_l2_coverage,
        "eligible_rows": int(len(eligible)),
        "best_policy": best["policy"],
        "best_strategy_type": best["strategy_type"],
        "best_ev_bps": float(best["event_ev_after_rebate_bps"]),
        "best_score_bps": float(best["implementation_score_bps"]),
    }


constraint_rows = []
for min_fill_rate in [0.0, 0.01, 0.02, 0.05, 0.08]:
    for max_ttl_ms in [60_000, 300_000, 900_000, 1_800_000]:
        for min_l2_coverage in [0.0, 0.01, 0.05, 0.20]:
            constraint_rows.append(best_under_constraints(min_fill_rate, max_ttl_ms, min_l2_coverage))

constraint_grid = pd.DataFrame(constraint_rows)
display(constraint_grid.sort_values(["min_l2_coverage", "min_fill_rate", "max_ttl_ms"]).head(80))


## Standalone Versus Supported Decomposition

The cleanest diagnostic is whether the model's standalone quote value is positive. If not, the quote is supported. A supported quote can still be valid, but only if the support term is observable before order send and survives event-level execution replay.


In [ ]:
support_decomposition = (
    policy_frontier[
        (policy_frontier["latency_ms"] == BASE_LATENCY_MS)
        & (policy_frontier["maker_rebate_bps"] == BASE_REBATE_BPS)
    ]
    .copy()
    .sort_values(["policy", "ttl_ms"])
)
support_decomposition["standalone_component_bps"] = support_decomposition["mean_predicted_standalone_bps"]
support_decomposition["support_component_bps"] = support_decomposition["support_gap_bps"]
support_decomposition["replay_component_bps"] = support_decomposition["event_ev_after_rebate_bps"]
support_decomposition["implementation_component_bps"] = support_decomposition["implementation_score_bps"]

display(
    support_decomposition[
        [
            "policy",
            "ttl_ms",
            "strategy_type",
            "standalone_component_bps",
            "support_component_bps",
            "replay_component_bps",
            "implementation_component_bps",
            "event_full_fill_rate_l2_queue",
            "l2_price_coverage_rate",
            "event_fills_l2_queue",
        ]
    ]
)


## Diagnostics

The charts prioritize implementability over raw backtest return: L2 queue-aware fill rate, EV after rebate, coverage, and the support gap.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

plot_frame = policy_frontier.copy()
plot_frame["ttl_label"] = (plot_frame["ttl_ms"] / 1000).astype(int).astype(str) + "s"

sns.scatterplot(
    data=plot_frame,
    x="event_full_fill_rate_l2_queue",
    y="event_ev_after_rebate_bps",
    hue="policy",
    style="strategy_type",
    size="ttl_minutes",
    sizes=(40, 260),
    ax=axes[0, 0],
)
axes[0, 0].axhline(0, color="black", linewidth=1)
axes[0, 0].axvline(PAPER_MIN_FILL_RATE, color="gray", linewidth=1, linestyle="--")
axes[0, 0].set_title("L2 Fill Rate Versus Event EV")

base_plot = policy_frontier[
    (policy_frontier["latency_ms"] == BASE_LATENCY_MS)
    & (policy_frontier["maker_rebate_bps"] == BASE_REBATE_BPS)
].copy()
sns.lineplot(data=base_plot, x="ttl_ms", y="implementation_score_bps", hue="policy", marker="o", ax=axes[0, 1])
axes[0, 1].axhline(0, color="black", linewidth=1)
axes[0, 1].set_xscale("log")
axes[0, 1].set_title("Implementation-Adjusted Score By TTL")

sns.lineplot(data=base_plot, x="ttl_ms", y="l2_price_coverage_rate", hue="policy", marker="o", ax=axes[1, 0])
axes[1, 0].axhline(LIVE_MIN_L2_COVERAGE, color="gray", linewidth=1, linestyle="--")
axes[1, 0].set_xscale("log")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_title("L2 Coverage By TTL")

component_plot = support_decomposition.melt(
    id_vars=["policy", "ttl_ms"],
    value_vars=["standalone_component_bps", "support_component_bps", "replay_component_bps", "implementation_component_bps"],
    var_name="component",
    value_name="bps",
)
sns.barplot(data=component_plot, x="policy", y="bps", hue="component", ax=axes[1, 1])
axes[1, 1].axhline(0, color="black", linewidth=1)
axes[1, 1].tick_params(axis="x", rotation=20)
axes[1, 1].set_title("Standalone, Support, Replay, And Implementation Components")

plt.tight_layout()


## PM Decision

Current evidence does **not** support a live standalone market-making strategy. The best-looking rows are better described as **supported paper candidates**, mainly because standalone quote value is not consistently positive and short TTL has too little capacity after L2 queue modeling.

The most defensible angle is:

1. Use market making as a passive execution layer for fair-value or inventory edge.
2. Require L2 queue-aware fill evidence, not just bar labels or trade-through without queue.
3. Paper trade the smallest allowed size while logging decision, send, ack, cancel ack, fills, market-data snapshot age, and quote replacement state.
4. Promote to live only after paper/live replay agrees with simulation on fill rate, slippage, and markout.

Supported market making is therefore a portfolio strategy, not a pure spread-capture strategy. It can be valid, but the support must be explicit before the order goes out.


## Optional Save

Set `SAVE_OUTPUTS = True` to write the scorecard, frontier, gate results, constraint grid, and support decomposition under `MODL_WS_FEATURE_DIR/quote_policy_optimizer/bar_<N>m`.


In [ ]:
if SAVE_OUTPUTS:
    out_dir = FEATURE_ROOT / "quote_policy_optimizer" / f"bar_{BAR_MINUTES}m"
    out_dir.mkdir(parents=True, exist_ok=True)
    angle_scorecard.to_parquet(out_dir / "angle_scorecard.parquet")
    policy_frontier.to_parquet(out_dir / "policy_frontier.parquet")
    gate_summary.to_parquet(out_dir / "gate_summary.parquet")
    paper_candidates.to_parquet(out_dir / "paper_candidates.parquet")
    live_candidates.to_parquet(out_dir / "live_candidates.parquet")
    research_candidates.to_parquet(out_dir / "research_candidates.parquet")
    constraint_grid.to_parquet(out_dir / "constraint_grid.parquet")
    support_decomposition.to_parquet(out_dir / "support_decomposition.parquet")
    print(f"wrote quote policy optimizer outputs to {out_dir}")
else:
    print("SAVE_OUTPUTS is false; nothing written")
